# 03 — Bronze Weather 2025 · 5 zones Taxi de Manhattan

**Périmètre choisi :** Times Square / Theatre District (230), Midtown Center (161), Upper East Side South (237), Penn Station / Madison Sq West (186), Financial District North (87).

Télécharge les réponses horaires **JSON brutes** de l’API Open-Meteo Historical Forecast, pour les 12 mois de 2025 et **5 points représentatifs** de ces zones TLC. Ce sont des points météo (coordonnées approximatives), **pas les polygones officiels** des zones Taxi.

- Aucune transformation Bronze ; conversion en Delta et jointure taxi/météo réservées à Silver/Gold.
- Un Lakehouse Fabric doit être **attaché et défini par défaut**.
- Chaque fichier existant est vérifié et ignoré s’il est valide.
- Le point Times Square reprend les coordonnées du précédent notebook `midtown_manhattan` et réutilise ses 12 JSON existants quand ils sont valides.
- Pour une évaluation ML rigoureuse, les *historical forecasts* ne garantissent pas toujours le bulletin réellement disponible à l’instant de prédiction : ce point devra être pris en compte pour prévenir toute fuite temporelle.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import calendar
import json
import shutil
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

YEAR = 2025
MONTHS = list(range(1, 13))  # Test rapide : [1]
TIMEZONE = "America/New_York"
API_URL = "https://historical-forecast-api.open-meteo.com/v1/forecast"
HOURLY = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "rain", "snowfall", "wind_speed_10m", "weather_code"
]
BASE = Path("/lakehouse/default/Files/bronze/weather/historical_forecast") / str(YEAR)
AUDIT = Path("/lakehouse/default/Files/bronze/_audit/weather")
BASE.mkdir(parents=True, exist_ok=True)
AUDIT.mkdir(parents=True, exist_ok=True)

# Coordonnées représentatives des 5 zones taxi (TLC). Vérifier les polygones
# TLC ultérieurement pour cartographier et effectuer les jointures spatiales.
LOCATIONS = [
    {"id": "times_square_theatre_district", "taxi_zone_id": 230, "name": "Times Square / Theatre District", "lat": 40.7580, "lon": -73.9855},
    {"id": "midtown_center", "taxi_zone_id": 161, "name": "Midtown Center", "lat": 40.7587, "lon": -73.9725},
    {"id": "upper_east_side_south", "taxi_zone_id": 237, "name": "Upper East Side South", "lat": 40.7683, "lon": -73.9630},
    {"id": "penn_station_madison_sq_west", "taxi_zone_id": 186, "name": "Penn Station / Madison Sq West", "lat": 40.7506, "lon": -73.9935},
    {"id": "financial_district_north", "taxi_zone_id": 87, "name": "Financial District North", "lat": 40.7078, "lon": -74.0092},
]
LEGACY_TIMES_SQUARE_DIR = BASE / "midtown_manhattan"
print("Lakehouse Files :", BASE)
print("Zones :", [(x["taxi_zone_id"], x["name"]) for x in LOCATIONS])

In [ ]:
retry = Retry(total=3, connect=3, read=3, backoff_factor=2,
              status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"], respect_retry_after_header=True)
session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=retry))
session.headers.update({"User-Agent": "Fabric-NYC-Taxi-Demand-Student-Project/1.0"})

def month_dates(year, month):
    first = f"{year}-{month:02d}-01"
    last = f"{year}-{month:02d}-{calendar.monthrange(year, month)[1]:02d}"
    return first, last

def destination(loc, month):
    return BASE / loc["id"] / f"weather_forecast_{YEAR}-{month:02d}_{loc['id']}.json"

def validate_raw_json(path, loc, month):
    """Check source response shape and time span; don't rewrite payload."""
    if not path.is_file() or path.stat().st_size < 100:
        return False, 0, "absent ou vide"
    try:
        raw = json.loads(path.read_bytes())
        hourly = raw.get("hourly") or {}
        dates = hourly.get("time") or []
        expected_first, expected_last = month_dates(YEAR, month)
        if not dates or not all(isinstance(t, str) for t in dates):
            return False, 0, "tableau horaire absent"
        if not dates[0].startswith(expected_first) or not dates[-1].startswith(expected_last):
            return False, len(dates), "période inattendue"
        # Accepter les transitions d'heure d'été à New York.
        expected_approx = calendar.monthrange(YEAR, month)[1] * 24
        if abs(len(dates) - expected_approx) > 2:
            return False, len(dates), "nombre d'heures inattendu"
        for variable in HOURLY:
            if variable not in hourly or len(hourly[variable]) != len(dates):
                return False, len(dates), f"variable manquante/incomplète : {variable}"
        if abs(float(raw.get("latitude", 0)) - loc["lat"]) > 0.25 or abs(float(raw.get("longitude", 0)) - loc["lon"]) > 0.25:
            return False, len(dates), "coordonnées incompatibles"
        return True, len(dates), "valide"
    except (OSError, ValueError, TypeError, KeyError) as exc:
        return False, 0, str(exc)

print("Fonctions de validation prêtes.")

In [ ]:
def fetch_month(loc, month):
    start, end = month_dates(YEAR, month)
    params = {
        "latitude": loc["lat"], "longitude": loc["lon"],
        "start_date": start, "end_date": end,
        "hourly": ",".join(HOURLY), "timezone": TIMEZONE,
    }
    response = session.get(API_URL, params=params, timeout=(20, 120))
    response.raise_for_status()
    # JSON brut tel que servi par Open-Meteo (pas de réencodage).
    payload = response.content
    target = destination(loc, month)
    target.parent.mkdir(parents=True, exist_ok=True)
    tmp = target.with_suffix(".json.part")
    tmp.write_bytes(payload)
    valid, hours, detail = validate_raw_json(tmp, loc, month)
    if not valid:
        tmp.unlink(missing_ok=True)
        raise ValueError(f"Réponse API incorrecte : {detail}")
    tmp.replace(target)
    return hours, len(payload), response.url

print("Téléchargement prêt.")

In [ ]:
# Ingestion réexécutable : vérifie, saute ou récupère chaque mois.
results = []
for loc in LOCATIONS:
    print(f"\n=== {loc['name']} (TLC {loc['taxi_zone_id']}) ===")
    for month in MONTHS:
        target = destination(loc, month)
        valid, hours, detail = validate_raw_json(target, loc, month)
        status = "SKIPPED" if valid else "DOWNLOADED"
        source_url = None
        error = None
        try:
            if not valid:
                # Reprise des fichiers déjà récupérés avec le précédent notebook.
                if loc["taxi_zone_id"] == 230:
                    legacy = LEGACY_TIMES_SQUARE_DIR / f"weather_forecast_{YEAR}-{month:02d}_midtown_manhattan.json"
                    old_ok, old_hours, _ = validate_raw_json(legacy, loc, month)
                    if old_ok:
                        target.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(legacy, target)  # Copie à l'identique
                        valid, hours, _ = validate_raw_json(target, loc, month)
                        status = "REUSED"
                if not valid:
                    hours, _, source_url = fetch_month(loc, month)
                    status = "DOWNLOADED"
                    time.sleep(0.5)  # Limiter les appels successifs
            print(f"  {month:02d} : {status} ({hours} heures)")
        except Exception as exc:
            status = "ERROR"
            error = str(exc)
            print(f"  {month:02d} : ERROR — {error[:180]}")
        results.append({
            "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            "year": YEAR, "month": month, "location_id": loc["id"],
            "taxi_zone_id": loc["taxi_zone_id"], "latitude": loc["lat"], "longitude": loc["lon"],
            "status": status, "hours": hours if status != "ERROR" else None,
            "path": str(target), "source_url": source_url, "error": error,
        })

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
audit_path = AUDIT / f"ingestion_weather_5_taxi_zones_{YEAR}_{run_id}.json"
audit_path.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nJournal d'ingestion :", audit_path)
print("Erreurs :", sum(x["status"] == "ERROR" for x in results))

In [ ]:
# Validation finale sur l'ensemble des 5 zones / 12 mois
summary = []
for loc in LOCATIONS:
    good = 0
    n_hours = 0
    bad = []
    for month in MONTHS:
        ok, hours, msg = validate_raw_json(destination(loc, month), loc, month)
        if ok:
            good += 1
            n_hours += hours
        else:
            bad.append(f"{month:02d}: {msg}")
    summary.append((loc["taxi_zone_id"], loc["name"], good, n_hours, bad))
    print(f"{loc['taxi_zone_id']:>3} | {loc['name']:<31} | {good}/{len(MONTHS)} fichiers | {n_hours} h")
    if bad:
        print("  PROBLÈMES :", "; ".join(bad))

total = sum(row[2] for row in summary)
expected = len(LOCATIONS) * len(MONTHS)
print(f"\nTOTAL : {total}/{expected} fichiers JSON valides")
assert total == expected, "Des fichiers sont absents ou invalides. Corriger les erreurs et relancer l'ingestion."
print("Bronze météo prêt pour Silver.")

## Suite du projet

Dans **Silver**, nous préparerons la météo horaire sans modifier les JSON Bronze. Pour toutes les zones TLC, prévoir une table de correspondance `PULocationID → point météo` fondée sur les polygones/zones du référentiel TLC et une règle de proximité explicite. Les cinq points collectés ici servent d'abord aux **cinq zones Taxi choisies**, et **ne couvrent pas automatiquement les 262 zones** du dataset.
